# Intrusion Detection System

## Setul de date : CIC-DDoS2019

### 1. Identificatori / Volum
* **Protocol:** numarul protocolului
* **Flow Duration:** durata fluxului
* **Total Fwd Packets:** numarul total de pachete trimise
* **Total Backward Packets:** numarul total de pachete primite
* **Fwd Packets Length Total:** volumul de date Forward
* **Bwd Packets Length Total:** volumul de date Backward
---
### 2. Dimensiunea Pachetelor
* **Fwd / Bwd Packet Length Min:** dimensiunea minima
* **Fwd / Bwd Packet Length Max:** dimensiunea maxima
* **Fwd / Bwd Packet Length Mean:** dimensiunea medie
* **Fwd / Bwd Packet Length Std:** deviatia standard
* **Packet Length Min / Max / Mean / Std:** pentru toate pachetele
* **Packet Length Variance:** varianta
* **Avg Packet Size:** dimeniunea medie
* **Avg Fwd / Bwd Segment Size:** dimensiunea medie pe TCP
---
### 3. Timpul
* **Flow IAT Mean / Std / Max / Min:** timpul dintre oricare 2 pachete consecutive
* **Fwd IAT Total:** suma timpilor de asteptare intre pachetele de forward
* **Fwd IAT Mean / Std / Max / Min:** timpul scurs intre pachetele forward
* **Bwd IAT Total:** suma timpilor de asteptare intre pachetele de backward
* **Bwd IAT Mean / Std / Max / Min:** timpul scurs intre pachetele backward
---
### 4. Viteza
* **Flow Bytes/s:** viteza totala in bytes
* **Flow Packets/s:** viteza totala in pachete/s
* **Fwd / Bwd Packets/s:** viteza de trimitere/primire
* **Down/Up Ratio:** raportul dintre traficul backward si cel forward
---
### 5. TCP Flags
* **FIN Flag Count:** steaguri FIN (cerere de inchidere)
* **SYN Flag Count:** steaguri SYN (cerere de initiere)
* **RST Flag Count:** steaguri RST (resetare)
* **PSH Flag Count:** steaguri PSH (cere trimiterea datelor)
* **ACK Flag Count:** steaguri ACK (confirmare)
* **URG Flag Count:** steaguri URG (date prioritare)
* **CWE Flag Count:** steagul Congestion Window Reduced (retea aglomerata)
* **ECE Flag Count:** steagul ECN-Echo (controlul congestiei)
* **Fwd / Bwd PSH Flags:** de cate ori s-a setat steagul PSH pe directia forward / backward
* **Fwd / Bwd URG Flags:** de cate ori s-a setat steagul URG pe directia forward / backward
---
### 6. Ferestre TCP
* **Fwd / Bwd Header Length:** lungimea header-elor trimise / primite
* **Init Fwd / Bwd Win Bytes:** dimensiunea initiala a ferestrei de receptie TCP
* **Fwd Act Data Packets:** numarul de pachete trimise care au continut date utile
* **Fwd Seg Size Min:** dimensiunea minima a antetului pe forward
---
### 7. Subflows
* **Subflow Fwd / Bwd Packets:** numarul mediu de pachete 
* **Subflow Fwd / Bwd Bytes:** numarul mediu de bytes 
* **Fwd / Bwd Avg Bytes/Bulk:** media octetilor transferati
* **Fwd / Bwd Avg Packets/Bulk:** media pachetelor transferate
* **Fwd / Bwd Avg Bulk Rate:** rata de transfer
---
### 8. Active vs. Idle
* **Active Mean / Std / Max / Min:** flux activ
* **Idle Mean / Std / Max / Min:** flux inactiv

## Procesarea datelor

In [1]:
import pandas as pd
import glob
import os

folder = 'date/'
files = glob.glob(os.path.join(folder, '*.parquet'))

files_train = [f for f in files if 'training' in f.lower()]
files_test = [f for f in files if 'testing' in f.lower()]

df_train = pd.concat([pd.read_parquet(f) for f in files_train], ignore_index= True)
df_train.to_parquet("dataset_TRAINING.parquet")
print(df_train.shape[0])

df_test = pd.concat([pd.read_parquet(f) for f in files_test], ignore_index= True)
df_test.to_parquet("dataset_TESTING.parquet")
print(df_test.shape[0])

df_train.describe().transpose().to_csv('stats.csv')
df_train.corr(numeric_only=True).to_csv('correlation_matrix.csv')

125170
306201


## Antrenarea modelului Isolation Forest si generarea arborilor in Verilog

In [5]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split

FEATURES = [
    'Packet Length Min',      # dimensiunea minima a pachetelor
    'Flow IAT Mean',          # media timpului de asteptare pentru pachetele client -> server
    'ACK Flag Count',         # acknowledgment
    'Fwd IAT Mean',           # media timpului de asteptare intre 2 pachete
    'Bwd Packet Length Min',  # dimensiunea minima a pachetului client -> server
    'URG Flag Count',         # urgent
    'Bwd Packet Length Mean', # media dimensiunii server -> client
    'Fwd Packet Length Min',  # dimensiunea minima a pachetului server -> client
    'Bwd IAT Min',            # timpul minim de asteptare intre pachetele backward
]

BIT_WIDTHS = {
    'Packet_Length_Min':      '[10:0]',  # max 1472
    'Flow_IAT_Mean':          '[15:0]',  # scalat /1024, max ~49999
    'ACK_Flag_Count':         '',        # flag 1 bit
    'Fwd_IAT_Mean':           '[15:0]',  # scalat /1024, max ~49999
    'Bwd_Packet_Length_Min':  '[10:0]',  # max 1460
    'URG_Flag_Count':         '',        # flag 1 bit
    'Bwd_Packet_Length_Mean': '[11:0]',  # max 1851
    'Fwd_Packet_Length_Min':  '[11:0]',  # max 2131
    'Bwd_IAT_Min':            '[7:0]',   # max 206
}

def prep_data(df):
    X = df[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0).copy() #pastrez doar feature-urile alese pentru antrenare

    # Fwd IAT Mean si Flow IAT Mean sunt in microsecunde (tranformate acm in milisecunde), se generau mai multe cazuri de false positive in varianta bruta
    X['Fwd IAT Mean'] = X['Fwd IAT Mean'] / 1024
    X['Flow IAT Mean'] = X['Flow IAT Mean'] / 1024

    X = X.astype(int)

    y = (~df['Label'].astype(str).str.upper().str.contains('BENIGN')).astype(int) #label-ul, 0 - normal, 1 - atac
    return X, y

train_df = pd.read_parquet('dataset_TRAINING.parquet')
test_df = pd.read_parquet('dataset_TESTING.parquet')

X_train_full, y_train_full = prep_data(train_df)
X_test, y_test = prep_data(test_df)

X_fit, X_val, y_fit, y_val = train_test_split(  # X/Y_fit pentru invatare 75%, X/Y_val pentru prag 25%
    X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=44  # stratify = pastreaza proportia prezenta in setul de date
)

X_fit_benign = X_fit[y_fit == 0]  # invata doar pe trafic normal

model = IsolationForest(n_estimators=4, max_samples=min(512, len(X_fit_benign)), random_state=44, n_jobs=-1)
# 4 arbori, alege maxim 512 de pachete pentru fiecare arbore, foloseste toate core-urile
model.fit(X_fit_benign)


#score_samples calculeaza scorul de anomalie bazat pe lungimea drumului
#returneaza valori negative, unde valorile spre -1 sunt anomalii, val spre 0 sunt normale
benign_scores = -model.score_samples(X_fit_benign)  # scorurile de anomalie a traficului normal
val_scores = -model.score_samples(X_val)          # scorurile de anomalie a setului de validare (trafic normal + atac)

best_config = {"f1": 0, "threshold": 0}

for q in np.linspace(0.50, 0.999, 500):  # 500 de praguri candidate, optimizez dupa F1 (echilibru intre a nu da alarme false si a nu rata cazurile reale)
    thr = np.quantile(benign_scores, q)       # sub = trafic normal; peste = trafic anormal (pe datele de trafic normal)
    y_val_pred = (val_scores >= thr).astype(int)  # aplic threshold-ul pe datele de validare,
    f1 = f1_score(y_val, y_val_pred, zero_division=0)  # F1 penalizeaza si falsele alarme, nu doar atacurile ratate

    if f1 > best_config["f1"]:
        best_config = {"threshold": thr, "f1": f1}

test_scores = -model.score_samples(X_test)
y_pred = (test_scores >= best_config["threshold"]).astype(int)

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()  # true negative, false positive, false negative, true positive

print("\n" + "="*55)
print("                   FINAL PERFORMANCE REPORT              ")
print("="*55)
print(f"OVERALL ACCURACY:       {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"F1 SCORE:               {f1_score(y_test, y_pred):.4f}\n")
print(f"ATTACKS DETECTED  (TP): {tp:>6} ({(tp/(tp+fn))*100:>5.2f}%)")
print(f"ATTACKS MISSED    (FN): {fn:>6} ({(fn/(tp+fn))*100:>5.2f}%)")
print(f"BENIGN ALLOWED    (TN): {tn:>6} ({(tn/(tn+fp))*100:>5.2f}%)")
print(f"FALSE ALARMS      (FP): {fp:>6} ({(fp/(tn+fp))*100:>5.2f}%)\n")
print("="*55 + "\n")

def walk_tree(tree, signal_names, node_id, current_depth, f):
    indent = "        " + "    " * current_depth

    if tree.feature[node_id] != -2: #-2 = frunza
        sig_name  = signal_names[tree.feature[node_id]]
        threshold = int(tree.threshold[node_id])
        bw_str = BIT_WIDTHS[sig_name]
        if bw_str != "":
            max_bits = int(bw_str.replace('[','').replace(']','').split(':')[0]) + 1 #nr max de biti pe care l poate avea feature ul
            max_val  = (1 << max_bits) - 1 #valoarea maxima
            if threshold > max_val:
                print(f"  Warning: {sig_name} threshold={threshold} exceeding {max_bits}-bit max={max_val} — threshold cut at {max_val}")
                threshold = max_val #daca are valoarea prea mare, este redus la maxim

        f.write(f"{indent}if ({sig_name} <= {threshold}) begin\n")
        walk_tree(tree, signal_names, tree.children_left[node_id],  current_depth + 1, f) #ramura din stanga
        f.write(f"{indent}end else begin\n")
        walk_tree(tree, signal_names, tree.children_right[node_id], current_depth + 1, f) #ramura din dreapta
        f.write(f"{indent}end\n")
    else: #frunza
        n_samples  = tree.n_node_samples[node_id]
        correction = (2.0 * (np.log(max(1, n_samples) - 1.0) + 0.5772156649)
                      if n_samples > 1 else 0.0) #estimeaza cat de lung ar fi fost drumul pentru a ajunge la pachet
        leaf_depth   = current_depth + correction
        scaled_depth = round(leaf_depth * 100)
        f.write(f"{indent}path_length = 16'd{scaled_depth};\n")


def export_verilog(model, features, threshold_value, filename="ids.v"):
    n = model.max_samples_
    c_n = 2.0 * (np.log(n - 1.0) + 0.5772156649) - (2.0 * (n - 1.0) / n)

    path_threshold = -np.log2(threshold_value) * c_n
    total_path_threshold = path_threshold * model.n_estimators #pentru a nu face impartirea in fpga
    INT_THRESHOLD = round(total_path_threshold * 100)

    signal_names = [col.replace(' ', '_').replace('/', '_') for col in features]

    with open(filename, "w") as f:
        for i, estimator in enumerate(model.estimators_):
            f.write(f"module Tree_{i} (\n")
            for sig in signal_names:
                bw = BIT_WIDTHS[sig]
                if bw == "":
                    f.write(f"    input       {sig},\n") # 1 bit
                else:
                    f.write(f"    input {bw} {sig},\n")
            f.write("    output reg [15:0] path_length\n") #blocuri always
            f.write(");\n")
            f.write("    always @(*) begin\n") #nu asteapta ciclul de ceas

            walk_tree(estimator.tree_, signal_names, 0, 0, f)

            f.write("    end\n")
            f.write("endmodule\n\n")

        f.write("module IDS_Top (\n")
        for sig in signal_names:
            bw = BIT_WIDTHS[sig]
            if bw == "":
                f.write(f"    input       {sig},\n")
            else:
                f.write(f"    input {bw} {sig},\n")
        f.write("    output alert\n")
        f.write(");\n\n")

        for i in range(model.n_estimators): #output-ul celor 4 arbori
            f.write(f"    wire [15:0] score_tree_{i};\n")
        f.write("\n")

        for i in range(model.n_estimators):
            f.write(f"    Tree_{i} t{i} (\n") #instantierea
            for sig in signal_names:
                f.write(f"        .{sig}({sig}),\n") #primul sig apartine modului instantiat, al 2 lea apartine IDS_Top
            f.write(f"        .path_length(score_tree_{i})\n")
            f.write("    );\n\n")

        f.write("    wire [31:0] total_score;\n")
        sum_logic = " + ".join([f"score_tree_{i}" for i in range(model.n_estimators)])
        f.write(f"    assign total_score = {sum_logic};\n\n")

        f.write(f"    assign alert = (total_score <= 32'd{INT_THRESHOLD}) ? 1'b1 : 1'b0;\n\n")
        f.write("endmodule\n")


export_verilog(model, FEATURES, best_config["threshold"])


                   FINAL PERFORMANCE REPORT              
OVERALL ACCURACY:       98.52%
F1 SCORE:               0.9911

ATTACKS DETECTED  (TP): 252353 (99.04%)
ATTACKS MISSED    (FN):   2444 ( 0.96%)
BENIGN ALLOWED    (TN):  49324 (95.95%)
FALSE ALARMS      (FP):   2080 ( 4.05%)




## Testare

In [2]:
import signal #semnale OS
import struct #transforma numerele in biti si octeti
import sys
import time #cronometrare rulare
from datetime import timedelta

import numpy as np
import pandas as pd
import serial #comunicare

DEFAULT_PORT = '/dev/ttyUSB0'
DEFAULT_BAUD = 115200
DEFAULT_DATASET = 'dataset_TESTING.parquet'
DEFAULT_DELAY = 0 
DEFAULT_SHUFFLE = False

HEADER = bytes([0xAA]) #fiecarui pachet ii adaug header-ul de 10101010 pentru sincronizarea ceasurilor si in cazul in care se pierd bytes

FEATURES = [
    'Packet Length Min',       
    'Flow IAT Mean',           
    'ACK Flag Count',          
    'Fwd IAT Mean',            
    'Bwd Packet Length Min',   
    'URG Flag Count',          
    'Bwd Packet Length Mean',  
    'Fwd Packet Length Min',   
    'Bwd IAT Min',             
]

IAT_SCALE_INDICES = {1, 3}  

def preprocess_row(row):
    values = []
    for i, feat in enumerate(FEATURES):
        v = row[feat]
        if not np.isfinite(v):
            v = 0.0
        v = int(v)
        if i in IAT_SCALE_INDICES:
            v = v // 1024
        v = max(0, v)
        values.append(v)
    return tuple(values)

def build_packet(values):
    return HEADER + struct.pack('>9I', *values) # > Big Endian, 9I - 9 Unsged Integers 4 bytes

def load_dataset(path, shuffle):
    df = pd.read_parquet(path)
    df['_label'] = (~df['Label'].astype(str).str.upper().str.contains('BENIGN')).astype(int) # 1 - atac, 0 - normal

    missing = [f for f in FEATURES if f not in df.columns]
    if missing:
        print(f"ERROR: Missing features in dataset: {missing}")
        sys.exit(1)

    if shuffle:
        df = df.sample(frac=1, random_state=44).reset_index(drop=True) #shuffle

    print(f"Dataset loaded: {len(df):,} rows.")
    return df

def print_final_report(stats, elapsed):
    total = stats['total']
    if total == 0:
        print("\nNo packets processed.")
        return

    tp, tn, fp, fn = stats['tp'], stats['tn'], stats['fp'], stats['fn']
    correct = tp + tn
    pps = total / elapsed if elapsed > 0 else 0.0
    elapsed_str = str(timedelta(seconds=int(elapsed)))

    print("\n" + "=" * 55)
    print("                 FINAL REPORT                     ")
    print("=" * 55)
    print(f"Total packets processed:  {total:>8,}")
    print(f"Timeouts:                 {stats['timeout']:>8,}")
    print()
    print(f"Total time:               {elapsed_str}  ({elapsed:.1f}s)")
    print(f"Average rate:             {pps:>8.1f} pkts/sec")
    print()
    print(f"Detected Attacks   (TP):  {tp:>8,}  ({tp/(tp+fn)*100:.2f}%)" if (tp+fn) > 0 else f"Detected Attacks   (TP):  {tp:>8,}")
    print(f"Missed Attacks     (FN):  {fn:>8,}  ({fn/(tp+fn)*100:.2f}%)" if (tp+fn) > 0 else f"Missed Attacks     (FN):  {fn:>8,}")
    print(f"Correct Normal     (TN):  {tn:>8,}  ({tn/(tn+fp)*100:.2f}%)" if (tn+fp) > 0 else f"Correct Normal     (TN):  {tn:>8,}")
    print(f"False Alarms       (FP):  {fp:>8,}  ({fp/(tn+fp)*100:.2f}%)" if (tn+fp) > 0 else f"False Alarms       (FP):  {fp:>8,}")
    print()
    print(f"Final Accuracy:           {correct/total*100:>8.2f}%")
    
    if (tp + fn) > 0:
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall    = tp / (tp + fn)
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        print(f"F1 Score:                 {f1:>8.4f}")
    print("=" * 55)

def run():
    df = load_dataset(DEFAULT_DATASET, DEFAULT_SHUFFLE)

    try:
        ser = serial.Serial(DEFAULT_PORT, DEFAULT_BAUD, timeout=2) #2 secunde timp pentru raspuns, astfel trece mai departe
        time.sleep(0.1)  
    except serial.SerialException as e:
        print(f"ERROR: Could not open serial port: {e}")
        sys.exit(1)

    stats   = {'total': 0, 'tp': 0, 'tn': 0, 'fp': 0, 'fn': 0, 'timeout': 0}
    running = True
    t_start = time.time()

    def handle_sigint(sig, frame):
        nonlocal running
        running = False

    signal.signal(signal.SIGINT, handle_sigint) #cand se opreste proramul, sare la handle_sigint, care seteaza running (cel global) la False, ca sa iasa din whiles

    idx = 0
    total_rows = len(df)

    try:
        while running and idx < total_rows:
            row = df.iloc[idx] #pachetul din setul de date
            expected = int(row['_label'])

            values = preprocess_row(row)
            packet = build_packet(values)
            ser.write(packet) #trimite pachetul modificat

            response = ser.read(1) #asteapta raspunsul
            stats['total'] += 1

            if response:
                fpga_decision = int.from_bytes(response, byteorder='big') #transforma impulsul electric in numar (0 sau 1)

                if fpga_decision == 1 and expected == 1:
                    stats['tp'] += 1
                elif fpga_decision == 0 and expected == 0:
                    stats['tn'] += 1
                elif fpga_decision == 1 and expected == 0:
                    stats['fp'] += 1
                else:
                    stats['fn'] += 1
            else:
                stats['timeout'] += 1
                print(f"[!] TIMEOUT at packet {stats['total']}")
                ser.reset_input_buffer()
                ser.reset_output_buffer()

            idx += 1
            time.sleep(DEFAULT_DELAY)

    except KeyboardInterrupt:
        pass
    finally: #executie garantata
        elapsed = time.time() - t_start
        ser.close() #opreste portul serial
        print_final_report(stats, elapsed)

run()

Dataset loaded: 306,201 rows.

                 FINAL REPORT                     
Total packets processed:   306,201
Timeouts:                        0

Total time:               0:28:55  (1735.7s)
Average rate:                176.4 pkts/sec

Detected Attacks   (TP):   252,353  (99.04%)
Missed Attacks     (FN):     2,444  (0.96%)
Correct Normal     (TN):    49,370  (96.04%)
False Alarms       (FP):     2,034  (3.96%)

Final Accuracy:              98.54%
F1 Score:                   0.9912
